In [ ]:
!apt-get update && apt-get install -y xvfb x11-utils
!pip install stable-baselines3 pyvirtualdisplay

In [ ]:
import os
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pygame

In [ ]:
# If running in Google Colab, set up a virtual display
try:
    from pyvirtualdisplay import Display
    _display = Display(visible=0, size=(400, 400))
    _display.start()
    print("Virtual display started for Colab.")
except ImportError:
    pass

In [ ]:
class SnakeEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 5}
    def __init__(self, grid_size=10, block_size=20, render_mode="rgb_array"):
        super().__init__()
        self.grid_size = grid_size
        self.block_size = block_size
        self.render_mode = render_mode
        # Action space
        self.action_space = spaces.Discrete(4)
        # Observation space
        self.obs_shape = (self.grid_size, self.grid_size, 3)
        self.observation_space = spaces.Box(low=0, high=255, shape=self.obs_shape, dtype=np.uint8)


        #Initialize pygame
        pygame.init()
        # Create off-screen surface for rgb_array
        self.screen = pygame.Surface((self.grid_size * self.block_size,
                                      self.grid_size * self.block_size))

        # For human mode, create a display window
        if self.render_mode == "human":
            self.window = pygame.display.set_mode((self.grid_size * self.block_size,
                                                   self.grid_size * self.block_size))
            pygame.display.set_caption("Snake")
            self.clock = pygame.time.Clock()


    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        pass
        return obs, info

    def step(self, action):
        if abs(action - self.direction) != 2:
            self.direction = action

        head_x, head_y = self.snake[0]
        if self.direction == 0:
            head_y -= 1
        elif self.direction == 1:
            head_x += 1
        elif self.direction == 2:
            head_y += 1
        elif self.direction == 3:
            head_x -= 1

        new_head = (head_x, head_y)
        reward = 0

        if (
            head_x < 0 or head_x >= self.grid_size
            or head_y < 0 or head_y >= self.grid_size
            or new_head in self.snake
        ):
            terminated = True
            truncated = False
            reward = -1



        return obs, reward, terminated, truncated, info

    def render(self, mode=None):
        pass

    def close(self):
        pass


In [ ]:
class SnakeEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 5}

    def __init__(self, grid_size=10, block_size=20, render_mode="rgb_array"):  # default to rgb_array for Colab
        super().__init__()
        self.grid_size = grid_size
        self.block_size = block_size
        self.render_mode = render_mode
        # Action space: 0=up, 1=right, 2=down, 3=left
        self.action_space = spaces.Discrete(4)
        # Observation: RGB image of the grid
        self.obs_shape = (self.grid_size, self.grid_size, 3)
        # self.obs_shape = (self.grid_size * self.block_size,
        #              self.grid_size * self.block_size,
        #              3)
        self.observation_space = spaces.Box(low=0, high=255, shape=self.obs_shape, dtype=np.uint8)

        # Initialize pygame for rendering
        pygame.init()
        # Create off-screen surface for rgb_array
        self.screen = pygame.Surface((self.grid_size * self.block_size,
                                      self.grid_size * self.block_size))
        # For human mode, create a display window
        if self.render_mode == "human":
            self.window = pygame.display.set_mode((self.grid_size * self.block_size,
                                                   self.grid_size * self.block_size))
            pygame.display.set_caption("Snake")
            self.clock = pygame.time.Clock()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        center = self.grid_size // 2
        self.snake = [(center, center)]
        self.direction = 1  # initial direction: right
        self._place_food()
        self.done = False
        self.time_step = 0
        obs = self._get_observation()
        return obs, {}

    def step(self, action):
        if abs(action - self.direction) != 2:
            self.direction = action

        head_x, head_y = self.snake[0]
        if self.direction == 0:
            head_y -= 1
        elif self.direction == 1:
            head_x += 1
        elif self.direction == 2:
            head_y += 1
        elif self.direction == 3:
            head_x -= 1

        #######################################################################################################################
        # old_distance_to_food = abs(head_x - self.food_pos[0])**2 + abs(head_y - self.food_pos[1])**2 # Calculate distance before move
        #######################################################################################################################
        new_head = (head_x, head_y)
        reward = 0

        if (
            head_x < 0 or head_x >= self.grid_size
            or head_y < 0 or head_y >= self.grid_size
            or new_head in self.snake
        ):
            self.done = True
            reward += -1
        else:
            self.snake.insert(0, new_head)
            if new_head == self.food_pos:
                reward = 10
                self._place_food()
            else:
                self.snake.pop()

                #######################################################################################################################
                # # Calculate distance reward after move (only if not dead and not eaten)
                # new_distance_to_food = abs(new_head[0] - self.food_pos[0])**2 + abs(new_head[1] - self.food_pos[1])**2
                # if new_distance_to_food < old_distance_to_food:
                #     reward += 0.01 # Reward for getting closer
                # elif new_distance_to_food > old_distance_to_food:
                #     reward -= 0.05 # Penalty for moving further away
                #######################################################################################################################

        obs = self._get_observation()

        return obs, reward, self.done, False, {}

    def _place_food(self):
        empty = [(x, y) for x in range(self.grid_size)
                 for y in range(self.grid_size)
                 if (x, y) not in self.snake]
        idx = self.np_random.choice(len(empty))
        self.food_pos = empty[idx]

    def _get_observation(self):
        obs = np.zeros((self.obs_shape[0], self.obs_shape[1], 3), dtype=np.uint8)
        # Draw food
        fx, fy = self.food_pos
        obs[fy, fx] = [255, 0, 0]
        # Draw snake
        for x, y in self.snake:
            obs[y, x] = [0, 255, 0]
        return obs

    def render(self, mode=None):
        mode = mode or self.render_mode
        # Draw to off-screen surface
        self.screen.fill((0, 0, 0))
        for y in range(self.grid_size):
            for x in range(self.grid_size):
                color = (0, 0, 0)
                if (x, y) == self.food_pos:
                    color = (255, 0, 0)
                elif (x, y) in self.snake:
                    color = (0, 255, 0)
                rect = pygame.Rect(x * self.block_size, y * self.block_size,
                                   self.block_size, self.block_size)
                pygame.draw.rect(self.screen, color, rect)

        if mode == "human":
            self.window.blit(self.screen, (0, 0))
            pygame.display.flip()
            self.clock.tick(self.metadata["render_fps"])
        elif mode == "rgb_array":
            arr = pygame.surfarray.array3d(self.screen)
            return np.transpose(arr, (1, 0, 2))  # HxWxC

    def close(self):
        pygame.quit()

In [ ]:
# train_agent.py
# Training script using Stable Baselines3 (PPO) on the SnakeEnv

from stable_baselines3 import PPO, DQN
from stable_baselines3.common.vec_env import DummyVecEnv, VecTransposeImage, VecFrameStack


def main():
    env = SnakeEnv(grid_size=10, block_size=20, render_mode="rgb_array")
    env = DummyVecEnv([lambda: env])
    # env = VecTransposeImage(env)
    # env = VecFrameStack(env, n_stack=2)
    # model = PPO('MlpPolicy', env, verbose=2)
    model = DQN('MlpPolicy', env, verbose=2)
    model.learn(total_timesteps=100_000)
    model.save('dqn_snake')

if __name__ == '__main__':
    main()

In [ ]:
# test_agent.py
# Script to load the trained agent and render inline in Google Colab

import time
import numpy as np
from IPython import display
import matplotlib.pyplot as plt
from stable_baselines3 import PPO, DQN
from stable_baselines3.common.vec_env import DummyVecEnv, VecTransposeImage, VecFrameStack

def video_from_frames(frames, fps=5):
    import matplotlib.animation as animation
    fig = plt.figure()
    plt.axis('off')
    im = plt.imshow(frames[0])
    def update(i):
        im.set_array(frames[i])
        return [im]
    anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=1000/fps)
    return anim


def main():
    # model = PPO.load('ppo_snake')
    model = DQN.load('dqn_snake')
    env = SnakeEnv(grid_size=10, block_size=20, render_mode="rgb_array")
    env = DummyVecEnv([lambda: env])
    # env = VecTransposeImage(env)
    # env = VecFrameStack(env, n_stack=2)
    obs = env.reset()
    done = False
    frames = []
    while not done:
        action, _ = model.predict(obs)
        obs, _, done, _ = env.step(action)
        frame = env.render(mode="rgb_array")
        frames.append(frame)
    env.close()
    anim = video_from_frames(frames, fps=5)
    display.display(display.HTML(anim.to_jshtml()))

if __name__ == '__main__':
    main()